In [ ]:
# Imports

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import sys
import os
import copy
import time

# Point to project root so we can import local modules
sys.path.append('..')

In [ ]:
#  Import from existing files

# Import model from models/efficientnet_model.py
from models.efficientnet_model import DeepfakeDetector

# Runs preprocessing.ipynb — loads train_loader, valid_loader, test_loader
%run preprocessing.ipynb

In [ ]:
# Device Setup

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# # Finding path to dataset

# import os

# for root, dirs, files in os.walk('/kaggle/input'):
#     level = root.replace('/kaggle/input', '').count(os.sep)
#     if level < 5:  # only show first 4 levels
#         print(root)

In [ ]:
# Model Setup

model = DeepfakeDetector(num_classes=2)
model = model.to(device)
print('Model loaded and moved to', device)

In [ ]:
# Freeze Backbone, Train Head Only First

# Freeze all backbone layers
for param in model.backbone.parameters():
    param.requires_grad = False

# Unfreeze only the classifier head
for param in model.backbone.classifier.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {trainable:,}')

In [ ]:
#  Loss, Optimizer, Scheduler

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=2, factor=0.1
)

In [ ]:
# Training Loop

EPOCHS = 10
best_val_acc = 0.0
best_model_weights = copy.deepcopy(model.state_dict())

train_losses, val_losses = [], []
train_accs,   val_accs   = [], []

os.makedirs('../saved_models', exist_ok=True)

for epoch in range(EPOCHS):
    start = time.time()

    # ── TRAIN ──────────────────────────────────────────
    model.train()
    train_loss, train_correct = 0.0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item() * images.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()

    # ── VALIDATE ───────────────────────────────────────
    model.eval()
    val_loss, val_correct = 0.0, 0

    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss    += loss.item() * images.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()

    # ── METRICS ────────────────────────────────────────
    train_loss /= len(train_data)
    val_loss   /= len(valid_data)
    train_acc   = train_correct / len(train_data) * 100
    val_acc     = val_correct   / len(valid_data) * 100

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    scheduler.step(val_loss)

    # ── SAVE BEST MODEL ────────────────────────────────
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_weights = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), '../saved_models/best_model.pth')
        print(f'  ✅ Best model saved — val acc: {val_acc:.2f}%')

    elapsed = time.time() - start
    print(f'Epoch [{epoch+1}/{EPOCHS}] '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% | '
          f'Time: {elapsed/60:.1f} min')

print(f'\nBest Validation Accuracy: {best_val_acc:.2f}%')
model.load_state_dict(best_model_weights)

In [ ]:
#  Plot Results

epochs_range = range(1, EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_range, train_losses, label='Train Loss')
ax1.plot(epochs_range, val_losses,   label='Val Loss')
ax1.set_title('Loss per Epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()

ax2.plot(epochs_range, train_accs, label='Train Acc')
ax2.plot(epochs_range, val_accs,   label='Val Acc')
ax2.set_title('Accuracy per Epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy %')
ax2.legend()

plt.tight_layout()
plt.savefig('../saved_models/training_curves.png')
plt.show()
print('Plot saved.')